In [14]:
import sys
sys.path.append("../")

import torch
from likelihoods.npll_torch import log_like_np
from models.psf import KingPSF
from utils.psf_correction import PSFCorrection

from jax.config import config
config.update("jax_enable_x64", True)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
kp = KingPSF()

pc_inst = PSFCorrection(delay_compute=True, num_f_bins=15)
pc_inst.psf_r_func = lambda r: kp.psf_fermi_r(r)
pc_inst.sample_psf_max = 10.0 * kp.spe * (kp.score + kp.stail) / 2.0
pc_inst.psf_samples = 10000
pc_inst.psf_tag = "Fermi_PSF_2GeV2"
pc_inst.make_or_load_psf_corr()

f_ary = pc_inst.f_ary
df_rho_div_f_ary = pc_inst.df_rho_div_f_ary

Loading the psf correction from: /net/rcstorenfs02/ifs/rc_labs/dvorkin_lab/smsharma/mi-attribution/notebooks/psf_dir/Fermi_PSF_2GeV2.npy


In [29]:
npix = 10

pt_sum_compressed = 10 * torch.randn((npix,))
npt_compressed = 10 * torch.randn((1, npix,))
data = torch.randint(20, (npix,))
f_ary = torch.Tensor(f_ary)
df_rho_div_f_ary = torch.Tensor(df_rho_div_f_ary)

theta = torch.Tensor([[0.1, 10, -2, -3, 10, 1]])

In [30]:
log_like_np(pt_sum_compressed, theta, npt_compressed, data, f_ary, df_rho_div_f_ary)

tensor([ -7.5657, -22.8589,  -4.7802,  -3.2860,   3.6016,  -4.5518,  -0.4036,
             nan, -15.4877,  -3.8144], dtype=torch.float64)

In [31]:
from likelihoods.npll_jax import log_like_np as log_like_np_jax

In [32]:
import jax.numpy as jnp

pt_sum_compressed = jnp.array(pt_sum_compressed, dtype=jnp.float64)
npt_compressed = jnp.array(npt_compressed, dtype=jnp.float64)
data = jnp.array(data, dtype=jnp.int64)
f_ary = jnp.array(f_ary, dtype=jnp.float64)
df_rho_div_f_ary = jnp.array(df_rho_div_f_ary, dtype=jnp.float64)

theta = jnp.array(theta, dtype=jnp.float64)

log_like_np_jax(pt_sum_compressed, theta, npt_compressed, data, f_ary, df_rho_div_f_ary)

DeviceArray([ -7.56570182, -22.85894535,  -4.78018382,  -3.28599093,
               3.60164868,  -4.5518093 ,  -0.40356487,          nan,
             -15.48770233,  -3.81437134], dtype=float64)